# c41 final — single training run

Trains **only** the winning phase-2 recipe:

```text
c41-lower-alph500-b3 : BPE + NFC + lowercase + WhitespaceSplit
+ alphabet 500 + byte fallback + ha/sw/yo/am x3
```

Expected validation result: **score 1.7440** (±0.005), guardrail **PASS**,
UNK = 0, vocab 10,000. The model is saved to `models/optimized_c41-*/` and
its reports to `reports/c41_final.*`.

Run end-to-end with `Runtime → Run all` (Google Colab, no GPU required).


## 1. Installation (official versions)

`tokenizers==0.22.1` is **required** by the challenge (the official checker
verifies the exact version match).


In [ ]:
!pip install -q "tokenizers==0.22.1" datasets pandas numpy sentencepiece
import tokenizers
print("tokenizers:", tokenizers.__version__, "(expected 0.22.1)")
assert tokenizers.__version__ == "0.22.1", "Install tokenizers==0.22.1 (official requirement)"

## 2. Official constants, metric and guardrail

Exact copy of the official metric (`competition/metrics.py`) and constants
(`competition/constants.py`) from the challenge repository.


In [ ]:
# =============================================================================
# 2. OFFICIAL constants + metric (competition)
# =============================================================================
import json, os, re, shutil, time, unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from tokenizers import Tokenizer

# ---- Official constants (competition/constants.py) ---------------------
LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
LANGUAGE_NAMES = {"en": "English", "fr": "French", "ha": "Hausa",
                  "sw": "Swahili", "yo": "Yoruba", "am": "Amharic"}
SCORED_LANGUAGES = ("ha", "sw", "yo", "am")
CONTEXT_LANGUAGES = ("en", "fr")
CONTEXT_FERTILITY_RATIO = 1.15
UNKNOWN_PENALTY = 100.0
MAX_VOCAB_SIZE = 10_000
MAX_TOKENIZER_BYTES = 20 * 1024 * 1024
REQUIRED_TOKENIZERS_VERSION = "0.22.1"
SMOKE_TEXTS = {
    "en": "Knowledge grows when it is shared.",
    "fr": "Le savoir grandit lorsqu’il est partagé.",
    "ha": "Ilimi yana ƙaruwa idan an raba shi.",
    "sw": "Maarifa hukua yanaposhirikishwa.",
    "yo": "Ìmọ̀ ń pọ̀ sí i nígbà tí a bá pín in.",
    "am": "እውቀት ሲካፈል ያድጋል።",
}

# ---- Official dataset ------------------------------------------------------
DATASET_NAME = "Similoluwa/african-multilingual-tokenizer-challenge"
DATASET_REVISION = "v1.0.0"

# ---- Run parameters (editable) ----------------------------------
VOCAB_SIZE = 10_000            # required by the challenge
MAX_TRAIN_DOCS = None          # None = all of train (240,000); e.g. 60,000 for a quick smoke run
BASELINE_REFERENCE_SCORE = 2.059977   # official score obtained by 01_baseline_bpe_10k

OUTPUT_ROOT = Path.cwd()
REPORT_DIR = OUTPUT_ROOT / "reports"
MODEL_DIR = OUTPUT_ROOT / "models"
SUBMISSIONS_DIR = OUTPUT_ROOT / "submissions"
for d in (REPORT_DIR, MODEL_DIR, SUBMISSIONS_DIR):
    d.mkdir(parents=True, exist_ok=True)


# ---- OFFICIAL METRIC ---------------------------------------------------
def count_words(text: str) -> int:
    """Words = whitespace-separated (like the official evaluator)."""
    return len(text.split())


def unknown_token_id(tokenizer: Tokenizer):
    """Id emitted for unrepresentable text (like the official evaluator)."""
    model = json.loads(tokenizer.to_str()).get("model", {})
    name = model.get("unk_token")
    if isinstance(name, str):
        return tokenizer.token_to_id(name)
    unk_id = model.get("unk_id")
    return int(unk_id) if unk_id is not None else None


def measure(rows, tokenizer, batch_size=2048):
    """rows = list of (language, text) -> fertility, unk_rate, tokens, words, lossy, unk_total."""
    tokens, words, unknowns = defaultdict(int), defaultdict(int), defaultdict(int)
    unknown_id = unknown_token_id(tokenizer)
    lossy = 0
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        encodings = tokenizer.encode_batch([t for _, t in batch], add_special_tokens=False)
        for (lang, text), enc in zip(batch, encodings, strict=True):
            tokens[lang] += len(enc.ids)
            words[lang] += count_words(text)
            if unknown_id is not None:
                unknowns[lang] += sum(1 for v in enc.ids if v == unknown_id)
            if tokenizer.decode(enc.ids, skip_special_tokens=False) != text:
                lossy += 1
    fertility = {l: tokens[l] / words[l] for l in LANGUAGES if words[l]}
    unk_rate = {l: unknowns[l] / words[l] for l in LANGUAGES if words[l]}
    return fertility, unk_rate, dict(tokens), dict(words), lossy, sum(unknowns.values())


def penalised_scores(fertility, unk_rate):
    return {l: v + UNKNOWN_PENALTY * unk_rate.get(l, 0.0) for l, v in fertility.items()}


def competition_score(fertility, unk_rate=None):
    """Official score = mean over scored languages (ha, sw, yo, am)."""
    missing = [l for l in SCORED_LANGUAGES if l not in fertility]
    if missing:
        raise ValueError(f"missing scored languages: {missing}")
    scores = penalised_scores(fertility, unk_rate or {})
    return sum(scores[l] for l in SCORED_LANGUAGES) / len(SCORED_LANGUAGES)


def guardrail(fertility):
    """budget = 1.15 x mean(raw fertility of scored languages)."""
    raw = sum(fertility[l] for l in SCORED_LANGUAGES) / len(SCORED_LANGUAGES)
    budget = raw * CONTEXT_FERTILITY_RATIO
    breaches = [l for l in CONTEXT_LANGUAGES if fertility.get(l, 0.0) > budget]
    return raw, budget, breaches


# ---- QUALITY METRIC (variant robustness, Kamali 2026) ----------------
# The official score cannot see WHERE the cuts fall: at close scores (±0.01),
# we select the tokenizer whose tokens survive meaning-preserving
# variants (case, NFD, multiple spaces, detached punctuation).
QUALITY_SEED = 7
QUALITY_SAMPLES_PER_LANG = 150
QUALITY_TIEBAND = 0.01           # band around the best raw score
QUALITY_VARIANTS = ("lower", "nfd", "dblspace", "detachpunct")

# Punctuation detached by the "detachpunct" variant (ASCII + quotes + Ethiopic)
_PUNCT_DETACH = (".,;:!?()[]{}" + "'" + '"'
                 + "\u00ab\u00bb\u2019\u2026\u2013\u2014"
                 + "\u1361\u1362\u1363\u1364\u1365\u1366\u1367\u1368")


def quality_samples(val_rows, per_lang=QUALITY_SAMPLES_PER_LANG, seed=QUALITY_SEED):
    """Deterministic sample (fixed seed) for the quality metric."""
    import random
    rng = random.Random(seed)
    by_lang = defaultdict(list)
    for lang, text in val_rows:
        by_lang[lang].append(text)
    samples = []
    for lang in LANGUAGES:
        pool = by_lang[lang]
        for i in rng.sample(range(len(pool)), min(per_lang, len(pool))):
            samples.append((lang, pool[i]))
    return samples


def _detach_punct(text):
    for p in _PUNCT_DETACH:
        if p in text:
            text = text.replace(p, f" {p} ")
    return text


def quality_variants(text):
    """4 meaning-preserving variants (raw text is the reference)."""
    return {
        "lower": text.lower(),
        "nfd": unicodedata.normalize("NFD", text),
        "dblspace": re.sub(r"\s", "  ", text),
        "detachpunct": _detach_punct(text),
    }


def _jaccard(ids_a, ids_b):
    set_a, set_b = set(ids_a), set(ids_b)
    if not set_a and not set_b:
        return 1.0
    return len(set_a & set_b) / len(set_a | set_b)


def measure_quality(tokenizer, samples):
    """Mean Jaccard (shared base/variant tokens) per variant + mean."""
    base_ids = [e.ids for e in tokenizer.encode_batch(
        [t for _, t in samples], add_special_tokens=False)]
    per_variant = {}
    for variant in QUALITY_VARIANTS:
        var_ids = [e.ids for e in tokenizer.encode_batch(
            [quality_variants(t)[variant] for _, t in samples], add_special_tokens=False)]
        per_variant[variant] = (sum(_jaccard(a, b) for a, b in zip(base_ids, var_ids))
                                / len(samples))
    per_variant["mean"] = sum(per_variant.values()) / len(per_variant)
    return per_variant


def validate_tokenizer_file(path):
    """Official validation checks (competition/validation.py)."""
    path = Path(path)
    checks, errors = {}, []
    checks["file_size"] = path.stat().st_size <= MAX_TOKENIZER_BYTES
    if not checks["file_size"]:
        errors.append("file > 20 MiB")
    tok = Tokenizer.from_file(str(path))
    checks["loads"] = True
    vocab_size = tok.get_vocab_size(with_added_tokens=True)
    checks["vocabulary"] = vocab_size <= MAX_VOCAB_SIZE
    if not checks["vocabulary"]:
        errors.append(f"vocabulary {vocab_size:,} > {MAX_VOCAB_SIZE:,}")
    encodings = tok.encode_batch(list(SMOKE_TEXTS.values()), add_special_tokens=False)
    checks["encodes_all_languages"] = all(e.ids for e in encodings)
    if not checks["encodes_all_languages"]:
        errors.append("one language produces no tokens")
    decoded = [tok.decode(e.ids, skip_special_tokens=False) for e in encodings]
    checks["decodes"] = all(t.strip() for t in decoded)
    if not checks["decodes"]:
        errors.append("one decoding is empty")
    import tokenizers as _tk
    checks["compatible_version"] = _tk.__version__ == REQUIRED_TOKENIZERS_VERSION
    if not checks["compatible_version"]:
        errors.append(f"tokenizers {_tk.__version__} != {REQUIRED_TOKENIZERS_VERSION}")
    lossy_languages = [l for l, orig, rest in zip(SMOKE_TEXTS, SMOKE_TEXTS.values(), decoded)
                       if orig != rest]
    return {"valid": all(checks.values()) and not errors, "checks": checks, "errors": errors,
            "vocab_size": vocab_size, "file_size_bytes": path.stat().st_size,
            "lossy_languages": lossy_languages}


print("Official metric loaded. Max vocab:", MAX_VOCAB_SIZE, "| guardrail ratio:", CONTEXT_FERTILITY_RATIO)
print("Tokenizers:", tokenizers.__version__ if (tokenizers := __import__("tokenizers")) else None)

## 3. Official data (train on `train` only)


In [ ]:
# =============================================================================
# 3. Loading the official dataset + preparing texts
# =============================================================================
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME, revision=DATASET_REVISION)
print(dataset)

train_by_lang = {}
for lang in LANGUAGES:
    train_by_lang[lang] = [t for t, l in zip(dataset["train"]["text"], dataset["train"]["language"]) if l == lang]
if MAX_TRAIN_DOCS:
    train_by_lang = {l: texts[:MAX_TRAIN_DOCS] for l, texts in train_by_lang.items()}

val_rows = []
for lang in LANGUAGES:
    texts = [t for t, l in zip(dataset["validation"]["text"], dataset["validation"]["language"]) if l == lang]
    val_rows.extend((lang, t) for t in texts)

print("\nTraining   :", {l: f"{len(v):,}" for l, v in train_by_lang.items()},
      "| total:", f"{sum(len(v) for v in train_by_lang.values()):,}")
print("Validation :", {l: f"{sum(1 for x, _ in val_rows if x == l):,}" for l in LANGUAGES},
      "| total:", f"{len(val_rows):,}")

quality_rows = quality_samples(val_rows)
print("Quality sample:", f"{len(quality_rows):,}", "texts (150/language x 6, fixed seed) -",
      len(QUALITY_VARIANTS), "variants per text")
assert len(dataset["train"]) == 240_000 or MAX_TRAIN_DOCS, "unexpected train"
assert len(val_rows) == 24_000, "unexpected validation"

## 4. c41 training (frozen recipe — do not modify)


In [ ]:
# =============================================================================
# c41-lower-alph500-b3 recipe — FROZEN, do not modify
#   BPE + NFC + lowercase + WhitespaceSplit + alphabet 500 + byte fallback
#   + scored languages x3. Expected validation score: 1.7440 (±0.005).
# =============================================================================
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.normalizers import NFC, Lowercase, Sequence as NormSequence
from tokenizers.pre_tokenizers import WhitespaceSplit
from tokenizers.decoders import ByteFallback
from tokenizers.trainers import BpeTrainer

BYTE_TOKENS = [f"<0x{i:02X}>" for i in range(256)]
BOOST = {"ha": 3, "sw": 3, "yo": 3, "am": 3}
EXPECTED_SCORE = 1.7440


def corpus_iterator(train_by_lang, boost):
    """Train texts in balanced round-robin, scored languages x3."""
    iters = {l: iter(texts) for l, texts in train_by_lang.items()}
    active = list(train_by_lang)
    i = 0
    while active:
        for lang in list(active):
            try:
                text = next(iters[lang])
            except StopIteration:
                active.remove(lang)
                continue
            for _ in range(boost.get(lang, 1)):
                yield text
                i += 1
                if i % 100_000 == 0:
                    print(f"    ... {i:,} texts yielded")


print("Training c41 on official train only (240,000 texts)...")
tokenizer = Tokenizer(BPE(unk_token="[UNK]", byte_fallback=True))
tokenizer.normalizer = NormSequence([NFC(), Lowercase()])
tokenizer.pre_tokenizer = WhitespaceSplit()
trainer = BpeTrainer(vocab_size=10_000, min_frequency=5,
                     special_tokens=["[UNK]"] + BYTE_TOKENS,
                     limit_alphabet=500)
t0 = time.time()
tokenizer.train_from_iterator(corpus_iterator(train_by_lang, BOOST), trainer=trainer)
print(f"Training done in {time.time() - t0:.1f} s")

# Plain byte tokens (Llama-2 form): removed from added_tokens, the
# byte_fallback flag is (re)forced — byte fallback intact, zero [UNK].
payload = json.loads(tokenizer.to_str())
payload["added_tokens"] = [t for t in payload["added_tokens"]
                            if not (len(t["content"]) == 6 and t["content"].startswith("<0x"))]
payload["model"]["byte_fallback"] = True
tokenizer = Tokenizer.from_str(json.dumps(payload))
tokenizer.decoder = ByteFallback()
print("Vocabulary:", tokenizer.get_vocab_size(with_added_tokens=True), "(expected 10000)")


## 5. Official evaluation on `validation`


In [ ]:
# =============================================================================
# Evaluation: official metric on validation + EN/FR guardrail
# =============================================================================
fertility, unk_rate, tokens, words, lossy, unk_total = measure(val_rows, tokenizer)
score = competition_score(fertility, unk_rate)
raw, budget, breaches = guardrail(fertility)
penalised = penalised_scores(fertility, unk_rate)

print(f"{'language':8s} {'fertility':>9s} {'unk_rate':>10s} {'score':>8s}")
for l in LANGUAGES:
    print(f"{l:8s} {fertility[l]:>9.4f} {unk_rate[l]:>10.6f} {penalised[l]:>8.4f}")
print(f"\nScore (mean ha/sw/yo/am): {score:.4f} "
      f"(expected {EXPECTED_SCORE:.4f}, delta {score - EXPECTED_SCORE:+.4f})")
print(f"Guardrail: budget {budget:.4f} | en {fertility['en']:.4f} | fr {fertility['fr']:.4f} "
      f"→ {'PASS' if not breaches else 'FAIL ' + str(breaches)}")
print(f"Total UNK: {unk_total} | lossy rows: {lossy:,} (whitespace/case dropped: normal)")
if abs(score - EXPECTED_SCORE) > 0.005:
    print("\n⚠️ ABNORMAL GAP (> 0.005) — investigate before submitting.")
else:
    print("\n✓ Score matches phase 2 (c41 = 1.7440).")


## 6. Saving the model + reports


In [ ]:
# =============================================================================
# Saving: models/optimized_c41-lower-alph500-b3/ + reports/c41_final.*
# =============================================================================
model_dir = MODEL_DIR / "optimized_c41-lower-alph500-b3"
model_dir.mkdir(parents=True, exist_ok=True)
tok_path = model_dir / "tokenizer.json"
tokenizer.save(str(tok_path))
print("Tokenizer saved:", tok_path, f"({tok_path.stat().st_size:,} bytes)")

candidate_path = OUTPUT_ROOT / "tokenizer.json"
shutil.copy2(tok_path, candidate_path)

report = {
    "experiment": "c41_final",
    "recipe": {"name": "c41-lower-alph500-b3", "model": "bpe", "normalizer": "nfc_lower",
               "pre": "whitespace_split", "min_freq": 5, "limit_alphabet": 500,
               "byte_fallback": True, "boost": BOOST},
    "dataset": {"name": DATASET_NAME, "revision": DATASET_REVISION,
                "train_texts": sum(len(v) for v in train_by_lang.values()),
                "validation_rows": len(val_rows)},
    "score": score,
    "expected_score": EXPECTED_SCORE,
    "fertility": fertility,
    "unk_rate": unk_rate,
    "penalised": penalised,
    "unk_total": unk_total,
    "lossy_rows": lossy,
    "guardrail": {"budget": budget, "breaches": breaches,
                  "pass": not breaches},
    "vocab_size": tokenizer.get_vocab_size(with_added_tokens=True),
    "environment": {"tokenizers": __import__("tokenizers").__version__,
                    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())},
}
(REPORT_DIR / "c41_final.json").write_text(json.dumps(report, ensure_ascii=False, indent=2),
                                           encoding="utf-8")
md = ["# c41 — final training (phase-2 recipe)", "",
      f"- Score: **{score:.4f}** (expected {EXPECTED_SCORE:.4f})",
      f"- Guardrail: {'PASS' if not breaches else 'FAIL ' + str(breaches)} "
      f"(budget {budget:.4f}, en {fertility['en']:.4f}, fr {fertility['fr']:.4f})",
      f"- UNK: {unk_total} | vocab: {tokenizer.get_vocab_size(with_added_tokens=True)}",
      f"- Model: `{tok_path}`", ""]
(REPORT_DIR / "c41_final.md").write_text("\n".join(md), encoding="utf-8")
print("Reports:", REPORT_DIR / "c41_final.json", "|", REPORT_DIR / "c41_final.md")


## 7. Validation with the challenge's **official checker**

Downloads `starter/utils.py` from the official repository and runs
`profile_submission` on the tokenizer — **the same code** used to validate
submissions.


In [ ]:
# =============================================================================
# 7. Official checker (starter/utils.py from the challenge repo)
# =============================================================================
OFFICIAL_UTILS_URL = ("https://raw.githubusercontent.com/aims-ai-research-foundations/"
                      "airf-multilingual-tokenizer-challenge/main/starter/utils.py")
utils_path = OUTPUT_ROOT / "utils.py"
if not utils_path.exists():
    import urllib.request
    try:
        urllib.request.urlretrieve(OFFICIAL_UTILS_URL, utils_path)
        print("official utils.py downloaded")
    except Exception as exc:
        print("Download failed:", exc)

official_report = None
if utils_path.exists():
    import importlib, sys
    sys.path.insert(0, str(OUTPUT_ROOT))
    import utils as official_utils
    importlib.reload(official_utils)
    official_report = official_utils.profile_submission(
        candidate_path, data=pd.DataFrame(val_rows, columns=["language", "text"]))
else:
    print("Official checker unavailable — using built-in checks.")
    print(json.dumps(validate_tokenizer_file(candidate_path), indent=2, ensure_ascii=False))

# Extra built-in checks (equivalent to competition/validation.py)
checks = validate_tokenizer_file(candidate_path)
print("\nBuilt-in official checks:", checks["checks"])
print("Errors:", checks["errors"] or "none")
print("Non-lossless languages (round-trip):", checks["lossy_languages"] or "none (lossless)")